# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [19]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [20]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [5]:
model_families = {
    "Sentiment classification": "Encoder-only (e.g. BERT, RoBERTa)",
    "Machine translation": "Encoder-decoder (e.g. T5, BART)",
    "Open-ended text generation": "Decoder-only (e.g. GPT)",
    "Named Entity Recognition": "Encoder-only (e.g. BERT, RoBERTa)",
    "Text summarization": "Encoder-decoder (e.g. T5, BART)",
}

for task, family in model_families.items():
    print(f"{task:30} -> {family}")

Sentiment classification       -> Encoder-only (e.g. BERT, RoBERTa)
Machine translation            -> Encoder-decoder (e.g. T5, BART)
Open-ended text generation     -> Decoder-only (e.g. GPT)
Named Entity Recognition       -> Encoder-only (e.g. BERT, RoBERTa)
Text summarization             -> Encoder-decoder (e.g. T5, BART)


- **Sentiment classification / NER → Encoder-only:** the model only needs to *understand* the input (a label for the whole text, or a label for each token), not generate new text.
- **Translation / summarization → Encoder–decoder:** an input sequence is transformed into a different output sequence; the encoder reads the input and the decoder writes the output.
- **Open-ended generation → Decoder-only:** the model keeps predicting the next token from the previous ones.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [6]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"{sentence:45} -> {result['label']:8} (score: {result['score']:.4f})")

The movie was excellent.                      -> POSITIVE (score: 0.9999)
The lecture was difficult to understand.      -> NEGATIVE (score: 0.9997)
I enjoyed learning about transformers.        -> POSITIVE (score: 0.9997)


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [7]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand, interpret, and generate human language. It is used in many applications such as "
    "machine translation, chatbots, sentiment analysis, and search engines. Modern NLP systems are "
    "based on transformer models, which are trained on large amounts of text and can learn the "
    "meaning of words from their context."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
# no_repeat_ngram_size stops the small model from repeating the same phrase in a loop
output = model.generate(**inputs, max_new_tokens=60, num_beams=4, no_repeat_ngram_size=3)

print("Summary:", tokenizer.decode(output[0], skip_special_tokens=True))

Summary: Learn about natural language processing.


**Note:** FLAN-T5-*small* has only ~77M parameters, so its summaries are very short and generic. Without `no_repeat_ngram_size=3` it repeated the same sentence until it reached the token limit. Larger versions (`google/flan-t5-base` or `google/flan-t5-large`) produce much better summaries with the same code.


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

**Answer:** A chatbot's job is to generate a reply word by word, and a decoder-only model is trained exactly for that: predicting the next token from all the previous tokens. The whole conversation (the user's messages and the bot's earlier replies) can be given as one long sequence of context, so the model does not need a separate encoder. Because it is trained on huge amounts of text with this simple next-token objective, it scales well and learns grammar, facts and reasoning patterns that make the replies fluent. Finally, the same model can handle many tasks (answering, explaining, writing code) just by changing the prompt, which is ideal for an open-ended assistant.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.

In [8]:
# Sampling cell: run it several times (without set_seed) and compare the outputs
# set_seed(42)

prompt = "Natural language processing is"

for i in range(3):
    output = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50)
    print(f"Run {i + 1}:", output[0]["generated_text"], "\n")

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is the preferred way to identify language input.





















 



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Natural language processing is not uncommon if you consider yourself a child-oriented, socialized family, or even part of your family group. As our parents often experience from other 



Run 3: Natural language processing is a highly sophisticated, dynamic language processing engine. It is not just an interactive language processing engine, it is a tool for the industry.

The 



**Answer:** With `do_sample=True`, the model does not always pick the most likely next token. It computes a probability for every possible token and then *randomly samples* one according to those probabilities. Each run uses different random numbers, so different tokens can be chosen, and every later token depends on the earlier choices, so the texts quickly become different. Calling `set_seed(42)` fixes the random number generator and makes the output repeatable, and `do_sample=False` (greedy decoding, as in Part A) always gives the same output.